In [20]:
from google.colab import files
uploaded = files.upload()

Saving Training_Files_final.xlsx to Training_Files_final.xlsx


In [21]:
import pandas as pd
df_train = pd.read_excel("Training_Files_final.xlsx")


In [22]:
import pandas as pd
import numpy as np

# ==============================
# Models
# ==============================
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet, HuberRegressor
)
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor

# ==============================
# Utilities
# ==============================
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# ==============================
# 1. From Laded Training Data
# ==============================

# Required columns
features = [
    'Annealing_Temperature [K]',
    'Eq_FCC',
    'Ms Temperature (K)',
]
target = 'Correction_Factor_MOD'

df_train = df_train[features + [target]].dropna()

X = df_train[features]
y = df_train[target]

# ==============================
# 2. TRAIN-TEST SPLIT
# ==============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.05, random_state=42
)

# ==============================
# 3. Define Models
# ==============================
models = {

    # Linear
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01),
    "ElasticNet": ElasticNet(alpha=0.01, l1_ratio=0.5),
    "Huber": HuberRegressor(),

    # Nonlinear



    "SVR": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR(C=10, epsilon=0.1))
    ]),

    "KNN": Pipeline([
        ("scaler", StandardScaler()),
        ("model", KNeighborsRegressor(n_neighbors=5))
    ])
}

# ==============================
# 4. Train & Select Best Model
# ==============================
best_model = None
best_r2 = -np.inf
best_name = ""

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    r2 = r2_score(y_test, y_pred)
    print(f"{name} R2: {r2:.4f}")

    if r2 > best_r2:
        best_r2 = r2
        best_model = model
        best_name = name

print("\nBest Model Selected:", best_name)

# ==============================
# 5. Retrain On Full Data
# ==============================
best_model.fit(X, y)



Linear Regression R2: 0.8064
Ridge R2: 0.7797
Lasso R2: 0.7504
ElasticNet R2: 0.7485
Huber R2: 0.5737
SVR R2: 0.0877
KNN R2: 0.6912

Best Model Selected: Linear Regression


LinearRegression()

In [23]:
from google.colab import files
uploaded = files.upload()

Saving Testing_File_final.xlsx to Testing_File_final.xlsx


In [24]:
# ==============================
# 6. Load new dataset for testing and prediction
# ==============================
#new_file = input("\nEnter path of NEW Excel file (only inputs): ")
df_new = pd.read_excel("Testing_File_final.xlsx")


# Ensure required columns exist
df_new = df_new[features]

# Handle missing values
df_new = df_new.fillna(method='ffill')

# ==============================
# 7. Predict correction factor
# ==============================
df_new['Predicted_Correction_Factor'] = best_model.predict(df_new)

# ==============================
# 8. Save output file in excel
# ==============================
output_file = input("\nEnter name for output Excel file: ")
df_new.to_excel(output_file, index=False)

/tmp/ipykernel_806/3045485345.py:12: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df_new = df_new.fillna(method='ffill')



Enter name for output Excel file: output_file_final.xlsx


In [25]:
# ==============================
# 6. Extended code for extracting empirical formula
# ==============================

if best_name == "Linear Regression":

    intercept = best_model.intercept_
    coefficients = best_model.coef_

    print("\n==============================")
    print("EMPIRICAL FORMULA FOR CF")
    print("==============================\n")

    print(f"CF = {intercept:.6f} "
          f"+ ({coefficients[0]:.6e}) * T(IA) "
          f"+ ({coefficients[1]:.6e}) * f_gamma_eq "
          f"+ ({coefficients[2]:.6e}) * Ms")

    print("\nWhere:")
    print("TIA        = Intercritical Annealing_Temperature [K]")
    print("f_gamma_eq = Eq_FCC")
    print("Ms         = Ms Temperature (K)")


EMPIRICAL FORMULA FOR CF

CF = 0.243186 + (-1.106371e-04) * T(IA) + (2.725765e-02) * f_gamma_eq + (-2.137061e-04) * Ms

Where:
TIA        = Intercritical Annealing_Temperature [K]
f_gamma_eq = Eq_FCC
Ms         = Ms Temperature (K)


In [26]:
# ==============================
# Final empirical equation
# ==============================

if best_name == "Linear Regression":

    b0 = best_model.intercept_
    b1, b2, b3 = best_model.coef_

    print("\n========================================")
    print("FINAL EMPIRICAL EQUATION")
    print("========================================\n")

    equation = (
        f"CF = {b0:.4f} "
        f"{b1:+.4e}·T(IA) "
        f"{b2:+.4e}·fγeq "
        f"{b3:+.4e}·Ms"
    )

    print(equation)

    print("\nVariables:")
    print("TIA  = Intercritical Annealing temperature (K)")
    print("fγeq = Equilibrium austenite fraction")
    print("Ms   = Martensite start temperature (K)")


FINAL EMPIRICAL EQUATION

CF = 0.2432 -1.1064e-04·T(IA) +2.7258e-02·fγeq -2.1371e-04·Ms

Variables:
TIA  = Intercritical Annealing temperature (K)
fγeq = Equilibrium austenite fraction
Ms   = Martensite start temperature (K)
